# 모델 개발·평가

Batch 1 정책 Hold-out만으로 최종 후보를 선택한다. 제공 라벨이 있는 Batch 2 39셀과 Batch 3 44셀을 평가한다. 결측 라벨 셀은 예측값과 무관하게 평가에서 제외한다. 전체 47셀 MAPE라고 표기하지 않는다.

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
project = Path.cwd() if (Path.cwd() / 'src').is_dir() else Path.cwd().parent
sys.path.insert(0, str(project))
from src.features import load_cells, feature_matrix
from src.preprocess import DQ_FEATURES, policy_holdout, grouped_cv
cells = load_cells()
labelled = cells[cells.label_status == 'provided'].copy()


## 외부·내부 정책 분리 확인

In [2]:
train, valid = policy_holdout(labelled[labelled.batch == 'Batch 1'])
groups = train.charging_policy.to_numpy()
X = feature_matrix(train); y = train.cycle_life.to_numpy()
assert not set(train.charging_policy) & set(valid.charging_policy)
for outer_train, outer_valid in grouped_cv(groups).split(X,y,groups):
    assert not set(groups[outer_train]) & set(groups[outer_valid])
    for inner_train, inner_valid in grouped_cv(groups[outer_train]).split(X[outer_train],y[outer_train],groups[outer_train]):
        assert not set(groups[outer_train][inner_train]) & set(groups[outer_train][inner_valid])
print('PASS: Hold-out 및 외부·내부 CV 정책 겹침 없음')

PASS: Hold-out 및 외부·내부 CV 정책 겹침 없음


## 학습과 성능표

내부 MAPE 튜닝: alpha 15개(10⁻⁴–10²), l1_ratio 0.1·0.5·0.9. 결측·스케일링·타깃 변환은 학습 fold 안에서 fit한다. 외부 CV는 학습 28셀 안에서만 계산한다.

In [3]:
from src.train import main
result = main()
display(result['performance'])
display(result['comparison'].round(2))
print('선택 파라미터:', result['model'].best_params_ if hasattr(result['model'],'best_params_') else 'fixed')

선택 모델: ElasticNet_log3_logtarget
분할: 28 8 39 44
                      구분  MAPE (%)                            비고
      Train (Batch 1 CV)      9.63      학습 분할 안 정책 GroupKFold 평균
Valid (Batch 1 Hold-out)      5.67 동일 프로토콜이 train/valid에 갈라지지 않음
          Test (Batch 2)     28.01            유효 라벨 39셀 / 전체 47셀
       Gap (Train-Valid)     -3.95                  (+) : 과적합 의심
        Gap (Valid-Test)     22.34           (+) : 배치간 일반화 저하 의심
       Gap (Target-Test)     18.91             Target : 원논문 9.1%
          Test (Batch 3)     14.09            유효 라벨 44셀 / 전체 46셀
     Gap (Batch2-Batch3)    -13.92                   두 테스트 배치 비교
       Gap (Target-Test)      4.99        Batch 3 기준, 논문 보고와의 차이
                      model  train_cv_mape  valid_mape  test_b2_mape  test_b3_mape
                  DummyMean          14.56       22.50         58.21         24.80
            ElasticNet_raw3           9.27       12.80         28.82         15.99
  ElasticNet_raw3_logtarget           8.76       12.0

,구분,MAPE (%),비고
0,Train (Batch 1 CV),9.63,학습 분할 안 정책 GroupKFold 평균
1,Valid (Batch 1 Hold-out),5.67,동일 프로토콜이 train/valid에 갈라지지 않음
2,Test (Batch 2),28.01,유효 라벨 39셀 / 전체 47셀
3,Gap (Train-Valid),-3.95,(+) : 과적합 의심
4,Gap (Valid-Test),22.34,(+) : 배치간 일반화 저하 의심
5,Gap (Target-Test),18.91,Target : 원논문 9.1%
6,Test (Batch 3),14.09,유효 라벨 44셀 / 전체 46셀
7,Gap (Batch2-Batch3),-13.92,두 테스트 배치 비교
8,Gap (Target-Test),4.99,"Batch 3 기준, 논문 보고와의 차이"


,model,train_cv_mape,valid_mape,test_b2_mape,test_b3_mape
0,DummyMean,14.56,22.50,58.21,24.80
1,ElasticNet_raw3,9.27,12.80,28.82,15.99
2,ElasticNet_raw3_logtarget,8.76,12.05,26.19,16.35
3,ElasticNet_logvar,8.81,9.86,28.72,12.48
4,ElasticNet_logvar_logtarget,9.02,9.30,29.90,12.10
5,ElasticNet_log3,9.53,7.66,26.47,14.27
6,ElasticNet_log3_logtarget,9.63,5.67,28.01,14.09
7,RandomForest,10.89,10.21,35.10,18.64
8,XGBoost,12.87,9.55,38.28,19.70


선택 파라미터: {'regressor__elasticnet__alpha': np.float64(0.0007196856730011522), 'regressor__elasticnet__l1_ratio': 0.1}


## 오류 분석

72.68%의 기존 평가는 결측 수명을 관측 종료 시점으로 채운 결과였다. 같은 XGBoost를 제공 라벨 39셀에서 계산하면 33.59%였다. 중도 종료된 Batch 1 10셀도 제외했다. ΔQ 로그 통계·로그 수명 ElasticNet은 Batch 1 Valid가 더 낮아 선택됐고, Batch 2 39셀에서 28.01%를 얻었다. Batch 2 성능만으로 가장 낮은 후보를 선택하지 않는다.

In [4]:
pred = result['predictions']
display(pred[pred.batch == 'Batch 2'].sort_values('abs_pct_error',ascending=False).head(10))
figure_dir = project / 'results' / 'figures'
figure_dir.mkdir(parents=True,exist_ok=True)
plt.rcParams.update({'font.size':13,'axes.titlesize':16,'axes.labelsize':14,'xtick.labelsize':12,'ytick.labelsize':12,'legend.fontsize':11,'figure.dpi':140,'savefig.dpi':200})
colors={'Valid (Batch 1 Hold-out)':'#1f4e79','Test (Batch 2)':'#c45c26','Test (Batch 3)':'#2d6a4f'}
fig,ax=plt.subplots(figsize=(8.4,5.8),constrained_layout=True)
for split,part in pred.groupby('split',sort=True):
    ax.scatter(part.cycle_life,part.prediction,s=76,alpha=.8,color=colors[split],edgecolor='white',linewidth=.7,label=f'{split} (n={len(part)})',zorder=3)
limit=float(max(pred.cycle_life.max(),pred.prediction.max()))
ax.plot([0,limit],[0,limit],ls='--',color='#444444',lw=1.5,label='Perfect prediction')
ax.set(xlabel='Actual cycle life (cycles)',ylabel='Predicted cycle life (cycles)',title='Selected model: predicted versus actual lifetime')
ax.legend(frameon=True,facecolor='white',edgecolor='none',loc='upper left')
fig.savefig(figure_dir/'fig3_pred_vs_actual.png',dpi=200,bbox_inches='tight');plt.show()
b2=pred[pred.batch=='Batch 2'].copy();b2['residual']=b2.prediction-b2.cycle_life
fig,ax=plt.subplots(figsize=(8.4,5.5),constrained_layout=True)
ax.scatter(b2.cycle_life,b2.residual,s=78,alpha=.83,color='#c45c26',edgecolor='white',linewidth=.7)
ax.axhline(0,ls='--',color='#444444',lw=1.5)
ax.set(xlabel='Actual cycle life (cycles)',ylabel='Prediction - actual (cycles)',title='Batch 2 residuals (39 labelled cells)')
fig.savefig(figure_dir/'fig4_batch2_residuals.png',dpi=200,bbox_inches='tight');plt.show()

,cell_id,batch,charging_policy,cycle_life,prediction,abs_pct_error,split
23,Batch 2-15,Batch 2,3.6C(9%)-5C,396.0,635.929028,60.588138,Test (Batch 2)
26,Batch 2-18,Batch 2,5.2C(50%)-4.25C,449.0,674.350517,50.189425,Test (Batch 2)
14,Batch 2-6,Batch 2,3.6C(9%)-5C,393.0,587.431956,49.473780,Test (Batch 2)
27,Batch 2-19,Batch 2,6C(60%)-3C,392.0,566.736811,44.575717,Test (Batch 2)
42,Batch 2-42,Batch 2,5.2C(50%)-4.25C,474.0,685.070264,44.529592,Test (Batch 2)
44,Batch 2-44,Batch 2,5.2C(58%)-4C-newstructure,841.0,1209.563854,43.824477,Test (Batch 2)
30,Batch 2-24,Batch 2,4.8C(80%)-4.8C,514.0,737.808649,43.542539,Test (Batch 2)
29,Batch 2-21,Batch 2,6C(60%)-3C,408.0,582.498459,42.769230,Test (Batch 2)
35,Batch 2-29,Batch 2,5.2C(58%)-4C,452.0,643.861681,42.447275,Test (Batch 2)
10,Batch 2-2,Batch 2,5.2C(50%)-4.25C,424.0,601.839820,41.943354,Test (Batch 2)


/var/folders/w7/5jl4lpk11hx7r83xkjybjmw00000gn/T/ipykernel_72256/150413541.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.savefig(figure_dir/'fig3_pred_vs_actual.png',dpi=200,bbox_inches='tight');plt.show()
/var/folders/w7/5jl4lpk11hx7r83xkjybjmw00000gn/T/ipykernel_72256/150413541.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.savefig(figure_dir/'fig4_batch2_residuals.png',dpi=200,bbox_inches='tight');plt.show()


## ESS 해석과 한계

Batch 2는 Valid보다 오차가 크다. 실험실 LFP 고속충전 데이터만으로 현장 교체 시점을 확정할 수 없다. 현장 부하·캘린더 에이징·온도 조건과 신규 배치에서 추가 검증이 필요하다. 30% 이하는 제공 라벨 39셀의 개발 결과이며, 라벨 없는 8셀의 수명 정확도는 확인할 수 없다.